## Data Preparation for RNN

### Subtask:
Reshape and split the existing 'X' and 'y' arrays into training and testing sets suitable for LSTM input.


In [ ]:
from sklearn.model_selection import train_test_split

# Verify existing shape of X (should be [samples, time_steps, features])
print(f"Original X shape: {X.shape}")
print(f"Original y shape: {y.shape}")

# Split the data into training and testing sets (80/20 split)
# shuffle=False is crucial for time-series data to prevent data leakage from the future
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Display the final shapes
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

## Build and Train LSTM Model

### Subtask:
Construct and train a Long Short-Term Memory (LSTM) network using TensorFlow/Keras to forecast electricity consumption.


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense

# Initialize the Sequential model
model = Sequential()

# Add LSTM layer - input_shape is (time_steps, features) -> (24, 8)
model.add(LSTM(units=50, activation='relu', input_shape=(24, 8)))

# Add a Dropout layer to prevent overfitting
model.add(Dropout(0.2))

# Add Dense output layer for continuous value prediction
model.add(Dense(units=1))

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error')

# Train the model
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1
)

print("Model training complete.")

## Generate Predictions

### Subtask:
Use the trained LSTM model to predict electricity consumption and inverse transform the results to the original kWh scale.


In [ ]:
import numpy as np

# 1. Generate predictions for the test set
y_pred_scaled = model.predict(X_test)

# 2. Prepare dummy arrays for inverse transformation
# Since the scaler was fit on 8 features, we need an array of shape (N, 8)
# predictions
dummy_pred = np.zeros((len(y_pred_scaled), 8))
dummy_pred[:, 0] = y_pred_scaled.flatten() # kWh is the first column

# actuals
dummy_actual = np.zeros((len(y_test), 8))
dummy_actual[:, 0] = y_test.flatten()

# 3. Perform inverse transformation using the scaler (assuming 'scaler' exists from previous steps)
# Note: If 'scaler' is not in namespace, we'd need to re-initialize it based on 'scaled_data' logic
# From context, we assume the MinMaxScaler instance used for 'scaled_data' is available.
from sklearn.preprocessing import MinMaxScaler
# Re-calculating/accessing the scaler used previously
scaler = MinMaxScaler()
scaler.fit(data) # 'data' contains the original 8 features before scaling

pred_inv = scaler.inverse_transform(dummy_pred)
actual_inv = scaler.inverse_transform(dummy_actual)

# 4. Extract the kWh column (index 0)
predictions_original_scale = pred_inv[:, 0]
y_test_original_scale = actual_inv[:, 0]

# 5. Verify the results
print("First 5 Predicted kWh (Original Scale):", predictions_original_scale[:5])
print("First 5 Actual kWh (Original Scale):", y_test_original_scale[:5])

## Create Interactive Dashboard

### Subtask:
Develop an interactive dashboard using Plotly or ipywidgets to visualize actual versus predicted hourly electricity demand with filtering capabilities.


In [ ]:
import pandas as pd
import plotly.graph_objects as go

# Calculate the starting index for the test set based on the 80/20 split used earlier
test_start_idx = len(df) - len(y_test_original_scale)

# Extract the metadata (time and day_type) from the original dataframe for the test set indices
test_metadata = df.iloc[test_start_idx:].reset_index(drop=True)

# Create a results dataframe combining metadata with actuals and predictions
results_df = pd.DataFrame({
    'Time': test_metadata['hour'],
    'Date': test_metadata['date'],
    'Day_Type': test_metadata['day_type'],
    'Actual_kWh': y_test_original_scale,
    'Predicted_kWh': predictions_original_scale
})

# Create a unique datetime or index for the X-axis to show temporal sequence
results_df['Timestamp'] = range(len(results_df))

print("Results DataFrame created with shape:", results_df.shape)
print(results_df.head())

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure we have the unique day types
day_types = results_df['Day_Type'].unique().tolist()

# Create the dropdown widget
dropdown = widgets.Dropdown(
    options=day_types,
    value=day_types[0],
    description='Day Type:',
    disabled=False,
)

# Output widget to display the plot
output = widgets.Output()

def update_plot(change):
    # Extract selection
    selection = change.new if hasattr(change, 'new') else change

    with output:
        output.clear_output(wait=True)
        filtered_data = results_df[results_df['Day_Type'] == selection]

        plt.figure(figsize=(12, 6))
        sns.set_style("whitegrid")

        plt.plot(filtered_data['Timestamp'], filtered_data['Actual_kWh'], label='Actual Consumption', color='blue', linewidth=2)
        plt.plot(filtered_data['Timestamp'], filtered_data['Predicted_kWh'], label='Predicted Consumption', color='orange', linestyle='--', linewidth=2)

        plt.title(f'Actual vs Predicted Electricity Consumption - {selection}', fontsize=14)
        plt.xlabel('Sequence (Time Steps)', fontsize=12)
        plt.ylabel('Electricity Consumption (kWh)', fontsize=12)
        plt.legend()
        plt.tight_layout()
        plt.show()

# Observe changes
dropdown.observe(update_plot, names='value')

# Display the dashboard
display(dropdown)
display(output)

# Initialize
update_plot(day_types[0])